# nnQC playground - liver / prostate / cardiac / spleen

One notebook to poke at the four trained nnQC models: reconstruct from a clean
mask, corrupt it at controlled severities, and watch the QC score track the
true Dice.

**Requires a GPU** (`nnqc.xa` is CUDA-only) - run this on a compute allocation,
e.g. `srun --pty -A vnc@h100 -C h100 --qos=qos_gpu_h100-dev --gpus-per-node=1 --time=01:00:00 jupyter lab`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch

import nnqc

ROOT = Path(os.environ.get("NNQC_ROOT", "/lustre/fsn1/projects/rech/rpv/commun/nnQC"))
os.chdir(ROOT)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "nnQC conditioning (xa) is CUDA-only"

: 

## The four tasks

Each entry points at its `configs/jz/<task>/{config,env}.json` pair and carries
the measured per-task DDIM step optimum. `cardiac_onehot` is the experimental
one-hot conditioning arm; flip `CARDIAC` to it to compare.

In [ ]:
TASKS = {
    "liver":    dict(config="configs/jz/liver/config.json",    env="configs/jz/liver/env.json",    num_steps=4),
    "prostate": dict(config="configs/jz/prostate/config.json", env="configs/jz/prostate/env.json", num_steps=4),
    "cardiac":  dict(config="configs/jz/cardiac/config.json",  env="configs/jz/cardiac/env.json",  num_steps=3),
    "spleen":   dict(config="configs/jz/spleen/config.json",   env="configs/jz/spleen/env.json",   num_steps=4),
    # experimental arm, same data as cardiac:
    "cardiac_onehot": dict(config="configs/jz/cardiac_onehot/config.json",
                           env="configs/jz/cardiac_onehot/env.json", num_steps=3),
}

def weights_ready(spec):
    import json as _json
    env = _json.load(open(ROOT / spec["env"]))
    md_ = Path(env["model_dir"])
    return all((md_ / f).exists() for f in
               ("autoencoder.pt", "diffusion_unet.pt", "xa.pt", "embed.pt"))

for name, spec in TASKS.items():
    print(f"{name:16s} steps={spec['num_steps']}  weights: {'OK' if weights_ready(spec) else 'not trained yet'}")

## 1. Pick a task and a validation volume

`val_volumes` reproduces the exact 80/20 split used in training, so these cases
were *not* trained on.

In [ ]:
from scripts.benchmark import corrupt_volume, val_volumes
from nnqc.config import resolve_config

TASK = "liver"            # <- liver | prostate | cardiac | spleen | cardiac_onehot
spec = TASKS[TASK]
assert weights_ready(spec), f"{TASK} weights not trained yet"
cfg = resolve_config(str(ROOT / spec["config"]), str(ROOT / spec["env"]), None, stage="diffusion")

vols = list(val_volumes(cfg))
print(f"{len(vols)} val volumes")
pair = vols[0]
print(pair["image"])

## 2. Clean mask vs corrupted masks

`corrupt_volume` stacks boundary corruption + slice dropout; measured on spleen,
benchmark severity 0.3 / 0.6 / 1.0 lands at foreground-slice Dice ~0.85 / ~0.55
/ ~0.33 (other organs are similar). The QC score should fall monotonically.

In [ ]:

def load_label_stack(path):
    g = np.asarray(nib.load(path).get_fdata())
    return torch.from_numpy(g.astype(np.float32)).permute(2, 0, 1).unsqueeze(1)  # [D,1,H,W]

gt = load_label_stack(pair["label"])
candidates = {"GT (sev 0)": pair["label"]}
import tempfile
tmp = Path(tempfile.mkdtemp())
for sev in (0.3, 0.6, 1.0):
    c = corrupt_volume(gt, cfg.num_classes, sev, seed=7)
    arr = c[:, 0].permute(1, 2, 0).numpy()
    ref = nib.load(pair["label"])
    out = tmp / f"corrupt_sev{sev}.nii.gz"
    nib.save(nib.Nifti1Image(arr, ref.affine, ref.header), out)
    candidates[f"sev {sev}"] = str(out)

results = {}
for name, mpath in candidates.items():
    r = nnqc.check(pair["image"], mpath, config=str(ROOT / spec["config"]),
                   env=str(ROOT / spec["env"]), num_steps=spec["num_steps"])
    results[name] = r
    td = r.true_dice if hasattr(r, "true_dice") else float("nan")
    print(f"{name:12s} QC score = {r.qc_score:.3f}")

## 3. Look at the pGTs

One row per candidate, mid-foreground slice: scan | candidate mask | nnQC
reconstruction (pGT). On near-blank candidates the model should lean on the
image and still propose a plausible organ (the fallback behaviour).

In [ ]:
def mid_slice(path):
    a = np.asarray(nib.load(path).get_fdata())
    fg = a.reshape(-1, a.shape[2]).sum(0)
    z = int(np.argmax(fg))
    return a, z

img3d = np.asarray(nib.load(pair["image"]).get_fdata())
if img3d.ndim == 4:
    img3d = img3d[..., 0]
n = len(results)
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
for i, (name, r) in enumerate(results.items()):
    recon_path = tmp / f"recon_{i}.nii.gz"
    r.save(str(recon_path))
    cand, z = mid_slice(candidates[name])
    rec, _ = mid_slice(str(recon_path))
    sl = img3d[..., z]
    for j, (data, ttl) in enumerate([(sl, "scan"),
                                     (cand[..., z], f"candidate {name}"),
                                     (rec[..., z], f"pGT (QC={r.qc_score:.2f})")]):
        ax = axes[i, j] if n > 1 else axes[j]
        ax.imshow(sl, cmap="gray") if j == 0 else ax.imshow(data)
        ax.set_title(ttl, fontsize=9); ax.axis("off")
fig.tight_layout()

## 4. Per-slice QC curve - localise the failure

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
for name, r in results.items():
    ax.plot(r.slice_ratios, r.slice_scores, "o-", ms=3, label=name)
ax.set_xlabel("slice ratio (0=apex, 1=base)")
ax.set_ylabel("per-slice QC score")
ax.legend()
fig.tight_layout()

## 5. Mini calibration sweep (one volume)

QC score vs true Dice across the severity ladder. The four-organ paper numbers
(band r, true Dice 0.1-0.9, GT==1.0 slices dropped): liver 0.82, prostate 0.55,
cardiac 0.53 (argmax) / 0.57 (one-hot). A single volume is illustrative only -
the real numbers need the full val set (`slurm/benchmark_nnqc.sh`).

In [ ]:
def dice_stack(a, b):
    inter = np.logical_and(a > 0, b > 0).sum()
    return 2 * inter / max((a > 0).sum() + (b > 0).sum(), 1)

gta = gt[:, 0].permute(1, 2, 0).numpy()
xs, ys = [], []
for sev in np.linspace(0.1, 1.0, 10):
    c = corrupt_volume(gt, cfg.num_classes, float(sev), seed=11)
    arr = c[:, 0].permute(1, 2, 0).numpy()
    out = tmp / f"sweep_{sev:.1f}.nii.gz"
    ref = nib.load(pair["label"])
    nib.save(nib.Nifti1Image(arr, ref.affine, ref.header), out)
    r = nnqc.check(pair["image"], str(out), config=str(ROOT / spec["config"]),
                   env=str(ROOT / spec["env"]), num_steps=spec["num_steps"])
    xs.append(dice_stack(arr, gta)); ys.append(r.qc_score)
    print(f"sev {sev:.1f}  true Dice {xs[-1]:.3f}  QC {ys[-1]:.3f}")

from scipy import stats
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(xs, ys, "o-")
ax.set_xlabel("true Dice (candidate vs GT)"); ax.set_ylabel("nnQC score")
r_ = stats.pearsonr(xs, ys)[0]
ax.set_title(f"{TASK} single-volume sweep  r={r_:.2f}")
fig.tight_layout()

## 6. Switch organs

Change `TASK` in section 1 and re-run from there. Same code works for all four
tasks - conditioning encoding (argmax vs one-hot) and step counts are read from
each task's config and checkpoint automatically.

## 7. Real-world masks: TotalSegmentator CT organs

So far every candidate mask came from corrupting ground truth. The actual use
case is the opposite: a segmenter produces a mask on a new scan and nnQC scores
it **without any ground truth**. Here we do exactly that - run
[TotalSegmentator](https://github.com/wasserth/TotalSegmentator), a strong
off-the-shelf CT segmenter, on one abdominal CT, score its spleen and liver
masks with the corresponding nnQC tasks, then corrupt those masks and watch the
QC score degrade.

**Requires a GPU** (`nnqc.xa` is CUDA-only, like the rest of this notebook) and
**internet access** for the first TotalSegmentator run (it downloads ~2 GB of
weights). On an offline cluster this section skips itself gracefully; run it on
a machine with network access.

In [ ]:
# Offline cluster note: this pip line only works where there is internet.
# Uncomment on a networked machine if totalsegmentator is missing:
# %pip install totalsegmentator

try:
    from totalsegmentator.python_api import totalsegmentator
    import totalsegmentator
    TS_OK = True
    print("totalsegmentator", totalsegmentator.__version__)
except ImportError as e:
    TS_OK = False
    print("totalsegmentator not installed - skipping this section (", e, ")")

In [ ]:
# One abdominal CT. Preferred: MONAI downloads a small public sample
# (MSD Task09_Spleen). Fallback: set LOCAL_CT to your own nifti path.
LOCAL_CT = None  # e.g. "/path/to/ct.nii.gz"

if TS_OK:
    ts_work = Path("qc_demo/totalseg"); ts_work.mkdir(parents=True, exist_ok=True)
    if LOCAL_CT is not None:
        ct_path = Path(LOCAL_CT)
    else:
        try:
            from monai.apps import DecathlonDataset
            ds = DecathlonDataset(root_dir=ts_work, task="Task09_Spleen",
                                  section="training", download=True,
                                  cache_rate=0.0, num_workers=0)
            ct_path = Path(ds[0]["image"]).resolve()
        except Exception as e:
            TS_OK = False
            print("could not download sample CT - skipping section (", e, ")")
            print("set LOCAL_CT to a local abdominal CT to run anyway")
if TS_OK:
    print("CT:", ct_path)

In [ ]:
# Run TotalSegmentator (task="total", fast mode) and extract organ masks by
# label id: spleen = 1, liver = 3.
if TS_OK:
    ts_out = ts_work / "ts_labels.nii.gz"
    if not ts_out.exists():
        try:
            totalsegmentator(str(ct_path), str(ts_out), fast=True, task="total")
        except Exception as e:  # e.g. weights cannot be downloaded offline
            TS_OK = False
            print("TotalSegmentator failed - skipping section (", e, ")")

if TS_OK:
    ts_img = nib.load(str(ts_out))
    ts_lab = np.asanyarray(ts_img.dataobj).astype(np.int16)
    TS_IDS = {"spleen": 1, "liver": 3}
    ts_masks = {}
    for organ, lid in TS_IDS.items():
        mp = ts_work / f"ts_{organ}.nii.gz"
        nib.save(nib.Nifti1Image((ts_lab == lid).astype(np.float32),
                                 ts_img.affine, ts_img.header), mp)
        ts_masks[organ] = str(mp)
        print(f"{organ}: {int((ts_lab == lid).sum())} voxels -> {mp}")

In [ ]:
# Score the TotalSegmentator masks with nnQC - no ground truth involved.
if TS_OK:
    ts_results = {}
    for organ, mpath in ts_masks.items():
        o_spec = TASKS[organ]
        assert weights_ready(o_spec), f"{organ} weights not trained yet"
        r = nnqc.check(str(ct_path), mpath,
                       config=str(ROOT / o_spec["config"]),
                       env=str(ROOT / o_spec["env"]),
                       num_steps=o_spec["num_steps"])
        ts_results[organ] = r
        print(f"{organ:8s} nnQC score = {r.qc_score:.3f}")

In [ ]:
# Mid-foreground slice: CT + TS mask (green contour) + nnQC reconstruction
# (orange contour) for each organ.
if TS_OK:
    ct3d = np.asarray(nib.load(str(ct_path)).get_fdata())
    if ct3d.ndim == 4:
        ct3d = ct3d[..., 0]
    fig, axes = plt.subplots(1, len(ts_results), figsize=(5 * len(ts_results), 5))
    axes = np.atleast_1d(axes)
    for ax, (organ, r) in zip(axes, ts_results.items()):
        recon_path = ts_work / f"recon_{organ}.nii.gz"
        r.save(str(recon_path))
        cand, z = mid_slice(ts_masks[organ])
        rec, _ = mid_slice(str(recon_path))
        ax.imshow(ct3d[..., z], cmap="gray",
                  vmin=np.percentile(ct3d, 1), vmax=np.percentile(ct3d, 99))
        ax.contour(cand[..., z], levels=[0.5], colors="lime", linewidths=1.2)
        ax.contour(rec[..., z], levels=[0.5], colors="orange", linewidths=1.2)
        ax.set_title(f"{organ}: TS mask (green) vs pGT (orange)\nQC={r.qc_score:.3f}",
                     fontsize=10)
        ax.axis("off")
    fig.tight_layout()

In [ ]:
# Degrade the TS masks with nnQC's anatomically realistic corruption
# (nnqc.corruptions.corrupt_ohe_masks_v2) at increasing severities - more
# passes = harsher boundary corruption. The QC score should drop.
if TS_OK:
    from nnqc.corruptions import corrupt_ohe_masks_v2

    torch.manual_seed(7)
    sevs = [0, 1, 2, 3]  # 0 = untouched TS mask; n>0 = n corruption passes
    fig, axes = plt.subplots(1, len(ts_results), figsize=(5.5 * len(ts_results), 3.6))
    axes = np.atleast_1d(axes)
    for ax, (organ, mpath) in zip(axes, ts_masks.items()):
        o_spec = TASKS[organ]
        ref = nib.load(mpath)
        stack = load_label_stack(mpath)  # [D,1,H,W], binary
        scores = [ts_results[organ].qc_score]
        ohe = (stack > 0.5).float()
        for sev in sevs[1:]:
            ohe = (corrupt_ohe_masks_v2(ohe, corruption_prob=1.0) > 0.5).float()
            arr = ohe[:, 0].permute(1, 2, 0).numpy()
            out = ts_work / f"ts_{organ}_corrupt_{sev}.nii.gz"
            nib.save(nib.Nifti1Image(arr, ref.affine, ref.header), out)
            r = nnqc.check(str(ct_path), str(out),
                           config=str(ROOT / o_spec["config"]),
                           env=str(ROOT / o_spec["env"]),
                           num_steps=o_spec["num_steps"])
            scores.append(r.qc_score)
            print(f"{organ:8s} sev {sev}  QC = {r.qc_score:.3f}")
        ax.plot(sevs, scores, "o-", color="tab:red", lw=2)
        ax.set_xlabel("corruption passes on TS mask")
        ax.set_ylabel("nnQC score")
        ax.set_ylim(-0.02, 1.02); ax.grid(alpha=0.3)
        ax.set_title(f"{organ}: QC score vs corruption")
    fig.tight_layout()